# Qwen3-Coder-30B-A3B 无审查版 · Colab 部署

在免费 T4 GPU 上运行 **Huihui-Qwen3-Coder-30B-A3B-Instruct-abliterated**，提供 OpenAI 兼容 API。

流程：配置 → 从 GitHub 克隆代码 → 下载预编译服务和 CUDA 库（约 2 分钟）→ 下载模型（约 14GB）→ 启动 API → 测试。

运行前请在 **代码执行程序 → 更改运行时类型** 中选择 **T4 GPU**。

**常见现象**：启动后立即请求会返回 503 `Loading model`（模型还在加载，等 30–60 秒）；首次生成请求因 CUDA 预热较慢，第二次起正常（预期生成 40–80 t/s）。验证 GPU：`!nvidia-smi` 应显示约 14GB 占用。

In [ ]:
# ===== 配置 =====
REPO_URL = "https://github.com/librook799-cpu/colab-qwen3-coder.git"  # 改成你的仓库地址
REPO_DIR = "colab-qwen3-coder"

QUANT = "Q3_K_M"   # 显存 OOM 时改成 "Q3_K_S"（13.3GB）
CTX = 8192          # 上下文长度，OOM 时降到 4096
PORT = 8000

In [ ]:
# ===== 从 GitHub 克隆代码 =====
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull || true

In [ ]:
# ===== 安装依赖（下载官方预编译 llama-server + CUDA 运行库，约 2 分钟，零编译）=====
import re, subprocess, threading, time

start = time.time()
status = {"line": "(准备中...)"}
done = {"flag": False}

def heartbeat():
    while not done["flag"]:
        el = int(time.time() - start)
        print(f"安装中 {el // 60:02d}:{el % 60:02d} | {status['line'][:80]}", flush=True)
        time.sleep(5)

threading.Thread(target=heartbeat, daemon=True).start()
proc = subprocess.Popen(["bash", "scripts/install.sh"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    line = line.rstrip()
    if re.search(r"\[\s*\d+[/%\]]|Collecting|Downloading|Installing|Successfully|校验|下载|ERROR|error:", line):
        status["line"] = line
rc = proc.wait()
done["flag"] = True
el = int(time.time() - start)
if rc == 0:
    print(f"安装完成，用时 {el // 60:02d}:{el % 60:02d}", flush=True)
else:
    print(f"安装失败 (exit={rc})，/content/build.log 末尾：", flush=True)
    try:
        with open("/content/build.log") as f:
            print("".join(f.readlines()[-30:]))
    except FileNotFoundError:
        pass
    raise RuntimeError("llama-server 安装失败")

In [ ]:
# ===== 下载模型（约 14GB，已下载则秒过）=====
!python scripts/download_model.py --quant {QUANT}

In [ ]:
# ===== 启动 OpenAI 兼容 API（后台）=====
import os, subprocess, time, urllib.request

subprocess.run(["pkill", "-f", "llama-server"], stderr=subprocess.DEVNULL)
time.sleep(2)

env = os.environ.copy()
env["MODEL_PATH"] = f"/content/models/Huihui-Qwen3-Coder-30B-A3B-Instruct-abliterated.i1-{QUANT}.gguf"
env["CTX"] = str(CTX)
env["PORT"] = str(PORT)

server_log = open("/content/server.log", "w")
proc = subprocess.Popen(["bash", "scripts/serve.sh"], env=env,
                        stdout=server_log, stderr=subprocess.STDOUT)

def _show_log():
    print("=" * 40, "server.log 末尾", "=" * 40)
    try:
        with open("/content/server.log") as f:
            print("".join(f.readlines()[-40:]))
    except FileNotFoundError:
        print("(日志文件不存在)")
    print("=" * 90)

for i in range(120):
    if proc.poll() is not None:
        _show_log()
        raise RuntimeError("服务进程已退出，见上方日志")
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2)
        print(f"API 已就绪: http://127.0.0.1:{PORT}/v1")
        break
    except Exception:
        time.sleep(3)
else:
    _show_log()
    raise TimeoutError("启动超时（进程还活着但端口不通），见上方日志")

In [ ]:
# ===== 测试请求（首次会预热 CUDA，较慢；看 timings 里的速度）=====
import requests

r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions",
                  json={"model": "local",
                        "messages": [{"role": "user", "content": "Write a Python quicksort with comments"}],
                        "max_tokens": 300},
                  timeout=300)
data = r.json()
if "error" in data:
    print(data["error"])
else:
    print(data["choices"][0]["message"]["content"])
    t = data.get("timings", {})
    print("\n=== 速度 ===")
    print(f"预填 {t.get('prompt_per_second', 0):.1f} t/s | 生成 {t.get('predicted_per_second', 0):.1f} t/s")
    print("（首次请求因 CUDA 预热偏慢，再跑一次看真实速度）")

## （可选）公网访问

Colab 无法直接暴露端口。需要外部访问时用 cloudflared 隧道，运行下面单元格后从输出中拿 `https://xxx.trycloudflare.com` 地址，调用时用 `/v1/chat/completions`。

In [ ]:
# ===== 可选：cloudflared 公网隧道 =====
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!nohup cloudflared tunnel --url http://127.0.0.1:{PORT} > /content/cf.log 2>&1 &
import time, re
time.sleep(8)
!grep -o 'https://[^ ]*trycloudflare.com' /content/cf.log | tail -1

In [ ]:
# ===== 停止服务 =====
import subprocess
r = subprocess.run(["pkill", "-f", "llama-server"])
try:
    proc.terminate()
except Exception:
    pass
print("服务已停止" if r.returncode == 0 else "没有在跑的服务")